In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import dspy
from typing import Literal
import mlflow
load_dotenv(override=True)
#load environment from .env file openrouterAPIkey
API_KEY = os.getenv("openrouter_api_key")

# dspy.configure_cache(
#     enable_disk_cache=False,
#     enable_memory_cache=False,
# )

trainset_path = "../validation/train_groundtruth.csv"
valset_path = "../validation/val_groundtruth.csv"
testset_path = "../validation/test_groundtruth.csv"

In [2]:
# import mlflow

# # Specify the tracking server URI, e.g. http://localhost:5000
# mlflow.set_tracking_uri("http://localhost:5000")
# # If the experiment with the name "traces-quickstart" doesn't exist, MLflow will create it
# mlflow.set_experiment("traces-quickstart")
# mlflow.dspy.autolog()

In [3]:
CONTEXT = """
You are a world-leading expert in medical literature analysis, specialising in Lyme disease, 
PTLDS, and CLD. Your goal is to classify abstracts regarding their stance on PTLDS or CLD 
using the definitions below. Ensure scientific neutrality and minimize bias.

Definitions:
1. Supports PTLDS: 
        - Attributes persistent symptoms after successful Lyme disease treatment to other
        factors like
        immune dysfunction, chronic fatigue syndrome, or depression.
        - Opposes the use of prolonged or repeated antibiotic treatments for PTLDS.
        - Suggests or implies that CLD lacks scientific support.
2. Supports CLD: 
        - A contested condition wherein some believe symptoms result from an ongoing or active
        *Borrelia
        burgdorferi* infection after standard treatment, often cited as requiring extended
        antibiotic therapy.
        - Suggests that
3. Neutral: 
        - Presents balanced arguments without clearly supporting or refuting either PTLDS or CLD.
        - Discusses aspects of Lyme disease that relate to both sides of the PTLDS and CLD
        debate without taking a clear stance.
        - No clear stance or suppor
4. Unrelated: 
        - The abstract does not mention PTLDS or CLD, or the surrounding debate at all.
        - Focuses on other aspects of Lyme disease, such as acute Lyme disease, vector control,
        ecological
        studies, epidemiology, or diagnostic methods without mentioning chronic conditions.
        - References CLD without any explicit or implied previous treatment.
5. Animal Study: 
        - The abstract pertains exclusively to animal models or non-human subjects withoutdirect implications for PTLDS or CLD in humans.
        - Does not involve human clinical data or direct conclusions about PTLDS or CLD in
        humans.
        - If the animal study provides evidence directly relevant to PTLDS or CLD in humans,
        classify it accordingly under "Supports PTLDS", "Supports CLD", or "Neutral".
Confidence Levels: High, Medium, Low.
"""
ORIGINIAL = """You are a world-leading expert in medical literature analysis and particularly in medical
literature, specialising in Lyme disease and the debates surrounding chronic Lyme disease
(CLD) and post-treatment Lyme disease syndrome (PTLDS). Your task is to evaluate a series
of scientific paper abstracts related to Lyme disease and determine each abstract’s stance
on PTLDS and CLD. This task requires scientific impartiality to minimize any bias in
classifying abstracts concerning disputed subjects such as CLD or PTLDS. The current
literature on this topic is polarised and can be expressed as follows:
"Some medical experts argue that post-treatment symptoms, experienced by a subset of
patients after completing standard antibiotic therapy, can be attributed to what they
define as Post-Treatment Lyme Disease Syndrome (PTLDS). According to some health
organisations, these symptoms--ranging from fatigue to cognitive impairment--are likely
caused by immune responses or tissue damage, rather than persistent infection. PTLDS
concerns patients who experience persistent symptoms for at least six months after the
completion of recommended treatment for Lyme disease.
Conversely, other organisations advocate for the recognition of chronic Lyme disease
(CLD), contending that ongoing infection or immune dysfunction may be responsible for
these symptoms and recommend extended antibiotic regimens, pointing to contested evidence
of patient improvement. CLD is a broader term that encompasses a range of unexplained
symptoms that patients attribute to a persistent Lyme disease infection
that survives recommended treatment. Both PTLDS and CLD involve persistent symptoms, such
as fatigue, pain, and neurological issues, and both face with diagnostic difficulties due
to nonspecific symptoms and lack of clear biomarkers, contributing to ongoing debates in
the medical community."
Based on the above context and the debate, your task is to classify each abstract and
determine each abstract’s explicit or implicit stance regarding CLD or PTLDS based on the
following classifications:
- Supports PTLDS
- Supports CLD
- Neutral
- Unrelated
- Animal Study
Definitions for Classification:
1. Supports PTLDS:
- Attributes persistent symptoms after successful Lyme disease treatment to other
factors like
immune dysfunction, chronic fatigue syndrome, or depression.
- Opposes the use of prolonged or repeated antibiotic treatments for PTLDS.
- Suggests or implies that CLD lacks scientific support.
2. Supports CLD:
- A contested condition wherein some believe symptoms result from an ongoing or active
*Borrelia
burgdorferi* infection after standard treatment, often cited as requiring extended
antibiotic therapy.
- Suggests that persistent symptoms may require prolonged or repeated antibiotic
treatment.
3. Neutral:
- Presents balanced arguments without clearly supporting or refuting either PTLDS or CLD.
- Discusses aspects of Lyme disease that relate to both sides of the PTLDS and CLD
debate without taking a clear stance.
- No clear stance or support or refutation is provided regarding PTLDS or CLD, either
explicitly or implicitly.
4. Unrelated:
- The abstract does not mention PTLDS or CLD, or the surrounding debate at all.
- Focuses on other aspects of Lyme disease, such as acute Lyme disease, vector control,
ecological
studies, epidemiology, or diagnostic methods without mentioning chronic conditions.
- References CLD without any explicit or implied previous treatment.
5. Animal Study:
- The abstract pertains exclusively to animal models or non-human subjects without
direct implications
for PTLDS or CLD in humans.
- Does not involve human clinical data or direct conclusions about PTLDS or CLD in
humans.
- If the animal study provides evidence directly relevant to PTLDS or CLD in humans,
classify it accordingly under "Supports PTLDS", "Supports CLD", or "Neutral".
Additionally, assign a Confidence Level to each classification you make: High, Medium, or
Low, based on how confident you are in the given classification.
- High: The chosen classification is indisputably true and cannot possibly be any of the
other options.
The abstract contains explicit wording relating to its stance.
- Medium: The chosen classification requires a mix explicit and implicit evidence in the
abstract. There is some degree of certainty, but the classification is inferred rather
than explicitly stated. Typicallyarises from indirect language, ambiguous terms, or
implied support for a stance.
- Low: The classification is uncertain due to lack of clear evidence or conflicting
information in the abstract, relying more on inference and contextual understanding.
Typically occurs when the abstract is vague, uses ambiguous language, or includes evidence
that could be interpreted in multiple ways. Reflects significant uncertainty in the
classification, but that the chosen classification is more likely than the others.
Also, provide a reason comprising 2-3 sentences for your classification above. Clearly
state your justification and if the abstract implicitly or explicitly supports PTLDS or
CLD in humans. When determining the abstract’s stance, consider both explicit statements
and implications of the abstracts given context and chosen topic. Look for language that
suggests support or opposition even if not directly stated.
Input:
For each abstract, you will receive an index number of the abstract, a paper title, and
the abstract text.
JSON Output Structure:
Your output must be a JSON object containing the following fields:
- ‘classification‘: One of "Supports PTLDS", "Supports CLD", "Neutral", "Unrelated", or
"Animal Study"
- ‘confidence‘: Confidence level in the classification ("High", "Medium", or "Low").
- ‘reason‘: Provide a 2-3 sentence justification for your classification.
"""
# --- your existing signature ---
class ClassifyLymeAbstract(dspy.Signature):
        """
        You are a world-leading expert in medical literature analysis and particularly in medical
        literature, specialising in Lyme disease and the debates surrounding chronic Lyme disease
        (CLD) and post-treatment Lyme disease syndrome (PTLDS). Your task is to evaluate a series
        of scientific paper abstracts related to Lyme disease and determine each abstract’s stance
        on PTLDS and CLD. This task requires scientific impartiality to minimize any bias in
        classifying abstracts concerning disputed subjects such as CLD or PTLDS. The current
        literature on this topic is polarised and can be expressed as follows:
        "Some medical experts argue that post-treatment symptoms, experienced by a subset of
        patients after completing standard antibiotic therapy, can be attributed to what they
        define as Post-Treatment Lyme Disease Syndrome (PTLDS). According to some health
        organisations, these symptoms--ranging from fatigue to cognitive impairment--are likely
        caused by immune responses or tissue damage, rather than persistent infection. PTLDS
        concerns patients who experience persistent symptoms for at least six months after the
        completion of recommended treatment for Lyme disease.
        Conversely, other organisations advocate for the recognition of chronic Lyme disease
        (CLD), contending that ongoing infection or immune dysfunction may be responsible for
        these symptoms and recommend extended antibiotic regimens, pointing to contested evidence
        of patient improvement. CLD is a broader term that encompasses a range of unexplained
        symptoms that patients attribute to a persistent Lyme disease infection
        that survives recommended treatment. Both PTLDS and CLD involve persistent symptoms, such
        as fatigue, pain, and neurological issues, and both face with diagnostic difficulties due
        to nonspecific symptoms and lack of clear biomarkers, contributing to ongoing debates in
        the medical community."
        Based on the above context and the debate, your task is to classify each abstract and
        determine each abstract’s explicit or implicit stance regarding CLD or PTLDS based on the
        following classifications:
        - Supports PTLDS
        - Supports CLD
        - Neutral
        - Unrelated
        - Animal Study
        Definitions for Classification:
        1. Supports PTLDS:
        - Attributes persistent symptoms after successful Lyme disease treatment to other
        factors like
        immune dysfunction, chronic fatigue syndrome, or depression.
        - Opposes the use of prolonged or repeated antibiotic treatments for PTLDS.
        - Suggests or implies that CLD lacks scientific support.
        2. Supports CLD:
        - A contested condition wherein some believe symptoms result from an ongoing or active
        *Borrelia
        burgdorferi* infection after standard treatment, often cited as requiring extended
        antibiotic therapy.
        - Suggests that persistent symptoms may require prolonged or repeated antibiotic
        treatment.
        3. Neutral:
        - Presents balanced arguments without clearly supporting or refuting either PTLDS or CLD.
        - Discusses aspects of Lyme disease that relate to both sides of the PTLDS and CLD
        debate without taking a clear stance.
        - No clear stance or support or refutation is provided regarding PTLDS or CLD, either
        explicitly or implicitly.
        4. Unrelated:
        - The abstract does not mention PTLDS or CLD, or the surrounding debate at all.
        - Focuses on other aspects of Lyme disease, such as acute Lyme disease, vector control,
        ecological
        studies, epidemiology, or diagnostic methods without mentioning chronic conditions.
        - References CLD without any explicit or implied previous treatment.
        5. Animal Study:
        - The abstract pertains exclusively to animal models or non-human subjects without
        direct implications
        for PTLDS or CLD in humans.
        - Does not involve human clinical data or direct conclusions about PTLDS or CLD in
        humans.
        - If the animal study provides evidence directly relevant to PTLDS or CLD in humans,
        classify it accordingly under "Supports PTLDS", "Supports CLD", or "Neutral".
        Additionally, assign a Confidence Level to each classification you make: High, Medium, or
        Low, based on how confident you are in the given classification.
        - High: The chosen classification is indisputably true and cannot possibly be any of the
        other options.
        The abstract contains explicit wording relating to its stance.
        - Medium: The chosen classification requires a mix explicit and implicit evidence in the
        abstract. There is some degree of certainty, but the classification is inferred rather
        than explicitly stated. Typicallyarises from indirect language, ambiguous terms, or
        implied support for a stance.
        - Low: The classification is uncertain due to lack of clear evidence or conflicting
        information in the abstract, relying more on inference and contextual understanding.
        Typically occurs when the abstract is vague, uses ambiguous language, or includes evidence
        that could be interpreted in multiple ways. Reflects significant uncertainty in the
        classification, but that the chosen classification is more likely than the others.
        Also, provide a reason comprising 2-3 sentences for your classification above. Clearly
        state your justification and if the abstract implicitly or explicitly supports PTLDS or
        CLD in humans. When determining the abstract’s stance, consider both explicit statements
        and implications of the abstracts given context and chosen topic. Look for language that
        suggests support or opposition even if not directly stated.
        Input:
        For each abstract, you will receive an index number of the abstract, a paper title, and
        the abstract text.
        JSON Output Structure:
        Your output must be a JSON object containing the following fields:
        - ‘classification‘: One of "Supports PTLDS", "Supports CLD", "Neutral", "Unrelated", or
        "Animal Study"
        - ‘confidence‘: Confidence level in the classification ("High", "Medium", or "Low").
        - ‘reason‘: Provide a 2-3 sentence justification for your classification.
        """
        #context: str = dspy.InputField(desc="Context and definitions for classification.", default=CONTEXT)
        title: str = dspy.InputField(desc="The title of the scientific paper.")
        abstract: str = dspy.InputField(desc="The abstract text of the paper.")

        classification: Literal[
                "Supports PTLDS",
                "Supports CLD",
                "Neutral",
                "Unrelated",
                "Animal Study"
        ] = dspy.OutputField(desc="""Classes
        """)
        confidence: Literal["High", "Medium", "Low"] = dspy.OutputField(
                desc="Confidence level in the classification."
        )
        
        reason: str = dspy.OutputField(
                desc="2–3 sentence justification for the classification, indicating explicit or implicit stance."
        )


In [ ]:
#only use of first rater for gold standard (should we mix up?)

df_train = pd.read_csv(trainset_path)
df_val = pd.read_csv(valset_path)
df_test = pd.read_csv(testset_path)
gold_standard_train = []
for _, row in df_train.iterrows():
    gold_standard_train.append(
        dspy.Example(
            # context = CONTEXT,
            title = row['title'],
            abstract=row['abstract'],
            decision=row['interrater_1_classification'],
            reasoning=row['interrater_1_reason'],
            confidence=row['interrater_1_confidence'],
        ).with_inputs("title","abstract"),
    )
gold_standard_val = []
for _, row in df_val.iterrows():
    gold_standard_val.append(
        dspy.Example(
            # context = CONTEXT
            title = row['title'],
            abstract=row['abstract'],
            decision=row['interrater_1_classification'],
            reasoning=row['interrater_1_reason'],
            confidence=row['interrater_1_confidence'],
        ).with_inputs("title","abstract"),
    )
gold_standard_test = []
for _, row in df_test.iterrows():
    gold_standard_test.append(
        dspy.Example(
            # context = CONTEXT,
            title = row['title'],
            abstract=row['abstract'],
            decision=row['interrater_1_classification'],
            reasoning=row['interrater_1_reason'],
            confidence=row['interrater_1_confidence'],
        ).with_inputs("title","abstract"),
    )

In [5]:
# --- GEPA-compatible metric with optional textual feedback ---
# 2b: Evaluation Metric

def gepa_feedback_metric(gold: dspy.Example,
                         pred: dspy.Prediction,
                         trace=None,
                         pred_name=None,
                         pred_trace=None):
    # simple exact-match score on the decision
    score = 1.0 if getattr(pred, "classification", None) == gold.decision else 0.0
    pred_class = getattr(pred, "classification", None)
    gold_class = gold.decision
    gold_reason = getattr(gold, "reason", None)
    # brief feedback that GEPA can reflect on
    if score == 1.0:
        fb = "Decision matches gold. Keep citing PICOS elements clearly."
    else:
        # Assuming your student output has 'reasoning', 'classification', and 'confidence'
# and your gold data has 'classification' and 'abstract' (as context)

        fb = (
            f"CLASSIFICATION FAILURE: Predicted label '{pred_class}' "
            f"but the Correct (Gold) label is '{gold_class}'.\n"
            
            # 1. The Core Error and Actionable Contrast
            f"INCORRECT REASONING: The model's reasoning was: \"{getattr(pred, 'reason', 'N/A')}\". "
            f"This reasoning failed to justify the correct label '{gold_class}'.\n"
            
            # 2. Domain-Specific Guidance (Why the Gold is Correct)
            f"GOLD CONTEXT: The gold standard belongs to '{gold_class}' because "
            f"the abstract contains specific key signals/phrases (e.g., '{gold_reason}'). " # Use gold.reason here as the key signal
            f"Focus on finding evidence for the correct class instead of misinterpreting or overlooking the text.\n"

            # 3. Confidence Check (Did the model know it was guessing?)
            f"CONFIDENCE LEVEL: The model reported a confidence of '{getattr(pred, 'confidence', 'N/A')}'. "
            f"The new prompt should aim to ensure '{gold_class}' is chosen with High Confidence."
        )

    # Return only the score for GEPA
    return dspy.Prediction(score=score, feedback=fb)

In [6]:
# student_llm_string = "openai/gpt-5-nano"
# reflector_lmm_string = "openai/gpt-5-nano"

# student_llm_string ="openai/gpt-oss-20b"
# reflector_lmm_string = "openai/gpt-oss-20b"
# student_llm_string = "openai/gpt-4o-mini"
# student_llm_string = "openai/gpt-5-mini"
# reflector_lmm_string = "openai/gpt-4o-mini"
# student_llm_string = "google/gemini-2.0-flash-001"
# reflector_lmm_string = "openai/gpt-5-mini"
student_llm_string = 'openrouter/x-ai/grok-4-fast'
reflector_lmm_string = 'openrouter/x-ai/grok-4-fast'
# student_llm_string = 'openrouter/google/gemini-2.0-flash-001'
# reflector_lmm_string = 'openrouter/google/gemini-2.0-flash-001'
# gpt-5-mini
# quen8b instruct
#remove / from string

screener_path = "../classifier/stuedent_"  + student_llm_string.replace("/", "_") + "reflactor"  +  reflector_lmm_string.replace("/", "_")+ "_original_gepa_auto_medium.json"
result_path = "../results/results_stuedent_" + student_llm_string.replace("/", "_") + "reflactor"+  reflector_lmm_string.replace("/", "_") + "_original_gepa_auto_medium.json"

In [7]:
# Optional separate reflector and student
# student used everywhere unless overridden
student_lm = dspy.LM(
    model=student_llm_string,       # e.g. "openrouter/google/gemini-2.0-flash-001"
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    # (plus any model_params like temperature, max_tokens, etc)
    # temperature=1.0, top_p=1.0, seed=42
    temperature=1.0, max_tokens = 20000,
)
dspy.configure(lm=student_lm)

# higher-diversity reflector for GEPA
reflector_lm = dspy.LM(
    model=reflector_lmm_string,       # e.g. "openrouter/google/gemini-2.0-flash-001"
    api_base="https://openrouter.ai/api/v1",
    api_key=API_KEY,
    # temperature=1.0, top_p=0.95, seed=123
    temperature=1.0, max_tokens = 20000,
    )

gepa = dspy.GEPA(
    metric=gepa_feedback_metric,      # your feedback metric
    reflection_lm=reflector_lm,  
    # only the reflector is stochastic
    auto = 'medium',
    reflection_minibatch_size=5,
    use_merge=True,
    max_merge_invocations=10,
    track_stats=True,
)

compiled_screener = gepa.compile(
    student=dspy.ChainOfThought(ClassifyLymeAbstract),
    trainset=gold_standard_train, # minimal viable setup
    valset=gold_standard_val,
)
cost = sum([x['cost'] for x in student_lm.history if x['cost'] is not None])  # cost in USD, as calculated by LiteLLM for certain providers
print(cost)

# --- save and load as before ---
compiled_screener.save(path=screener_path)
# same lm used for a minimal setup in this example

2025/11/10 16:31:45 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 780 metric calls of the program. This amounts to 10.83 full evals on the train+val set.
2025/11/10 16:31:45 INFO dspy.teleprompt.gepa.gepa: Using 18 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget.
GEPA Optimization:   0%|          | 0/780 [00:00<?, ?rollouts/s]2025/11/10 16:32:00 INFO dspy.evaluate.evaluate: Average Metric: 13.0 / 18 (72.2%)
2025/11/10 16:32:00 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.7222222222222222
GEPA Optimization:   2%|▏         | 18/780 [00:14<10:10,  1.25rollouts/s]2025/11/10 16:32:00 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.7222222222222222


Average Metric: 2.00 / 5 (40.0%): 100%|██████████| 5/5 [00:06<00:00,  1.29s/it]

2025/11/10 16:32:06 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 5 (40.0%)


2025/11/10 16:32:15 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for predict: You are a world-leading expert in medical literature analysis, specializing in Lyme disease and the debates surrounding chronic Lyme disease (CLD) and post-treatment Lyme disease syndrome (PTLDS). Your task is to evaluate scientific paper abstracts related to Lyme disease and classify each abstract’s explicit or implicit stance on PTLDS and CLD with scientific impartiality to minimize bias. The literature is polarized: PTLDS attributes persistent symptoms (e.g., fatigue, cognitive impairment, pain, neurological issues for at least six months post-treatment) to non-infectious causes like immune responses or tissue damage, opposing prolonged antibiotics; CLD posits ongoing Borrelia burgdorferi infection or immune dysfunction causing these symptoms, advocating extended antibiotic therapy based on contested evidence of improvement. Both involve nonspecific symptoms and diagnostic challenges with

Average Metric: 4.00 / 5 (80.0%): 100%|██████████| 5/5 [00:11<00:00,  2.21s/it] 

...

2025/11/10 16:51:02 INFO dspy.teleprompt.gepa.gepa: Iteration 34: Proposed new text for predict: You are a world-leading expert in medical literature analysis, specializing in Lyme disease and the debates surrounding chronic Lyme disease (CLD) and post-treatment Lyme disease syndrome (PTLDS). Your task is to evaluate scientific paper abstracts related to Lyme disease and classify each abstract’s explicit or implicit stance on PTLDS and CLD with scientific impartiality to minimize bias. The literature is polarized: PTLDS attributes persistent symptoms (e.g., fatigue, cognitive impairment, pain, neurological issues for at least six months post-treatment) to non-infectious causes like immune responses or tissue damage, opposing prolonged antibiotics; CLD posits ongoing Borrelia burgdorferi infection or immune dysfunction causing these symptoms, advocating extended antibiotic therapy based on contested evidence of improvement. Both involve nonspecific symptoms and diagnostic challenges wit

Average Metric: 4.00 / 5 (80.0%): 100%|██████████| 5/5 [00:00<00:00, 309.94it/s]

2025/11/10 16:51:30 INFO dspy.evaluate.evaluate: Average Metric: 4.0 / 5 (80.0%)


2025/11/10 16:51:42 INFO dspy.teleprompt.gepa.gepa: Iteration 35: Proposed new text for predict: You are a world-leading expert in medical literature analysis, specializing in Lyme disease and the debates surrounding chronic Lyme disease (CLD) and post-treatment Lyme disease syndrome (PTLDS). Your task is to evaluate scientific paper abstracts related to Lyme disease and classify each abstract’s explicit or implicit stance on PTLDS and CLD with scientific impartiality to minimize bias. The literature is polarized: PTLDS attributes persistent symptoms (e.g., fatigue, cognitive impairment, pain, neurological issues for at least six months post-treatment) to non-infectious causes like immune responses or tissue damage, opposing prolonged antibiotics; CLD posits ongoing Borrelia burgdorferi infection or immune dysfunction causing these symptoms, advocating extended antibiotic therapy based on contested evidence of improvement. Both involve nonspecific symptoms and diagnostic challenges wit

0.6555572500000001


In [ ]:
print(compiled_screener.detailed_results)

In [9]:
token_completioncounter = 0
totaltokens = 0
prompttokens = 0
counter = 0
for x in student_lm.history:
    counter = counter + 1
    if x['usage'] :
       
        token_completioncounter = token_completioncounter + x['usage']["completion_tokens"]
        totaltokens = totaltokens + x['usage']["total_tokens"]
        prompttokens = prompttokens + x['usage']["prompt_tokens"]
print(token_completioncounter)
print(totaltokens)
print(prompttokens)
print(counter)

727415
2761869
2034454
791


In [10]:
token_completioncounter = 0
totaltokens = 0
prompttokens = 0
counter = 0
for x in reflector_lm.history:
    counter = counter + 1
    if x['usage'] :
       
        token_completioncounter = token_completioncounter + x['usage']["completion_tokens"]
        totaltokens = totaltokens + x['usage']["total_tokens"]
        prompttokens = prompttokens + x['usage']["prompt_tokens"]
print(token_completioncounter)
print(totaltokens)
print(prompttokens)
print(counter)
cost = sum([x['cost'] for x in reflector_lm.history if x['cost'] is not None])  # cost in USD, as calculated by LiteLLM for certain providers
print(cost)
cost = sum([x['cost'] for x in student_lm.history if x['cost'] is not None])  # cost in USD, as calculated by LiteLLM for certain providers
print(cost)

66145
213333
147188
28
0.0616191
0.6555572500000001


In [11]:

loaded_screener = dspy.ChainOfThought(ClassifyLymeAbstract)
loaded_screener.load(screener_path)



# loaded_screener.load("../classifier/stuedent_reflactorclassify_abstract.json")
# test with testset
test_results = []
df_total = pd.concat([df_train, df_val, df_test], ignore_index=True)
# df_total = df_val
for _, row in df_total.iterrows():
    prediction = loaded_screener(
        # context = CONTEXT,
        title = row['title'],
        abstract=row['abstract'],
        
    )
    test_results.append({
        "abstract": row['abstract'],
        "predicted_decision": prediction.classification,
        "predicted_reasoning": prediction.reasoning,
        "predicted_confidence": prediction.confidence,
        "ground_truth_decision": row['interrater_1_classification'],
        "predicition_correct": prediction.classification == row['interrater_1_classification'],
    
    })
    # print(prediction.classification == row['interrater_1_classification'])
    
score = sum([1 if result['predicition_correct'] else 0 for result in test_results]) / len(test_results)
result = {"accuracy": score,
"results": test_results
}
# result_path = "../results/results_stuedent_" + student_llm_string.replace("/", "_") + "_original_notraining.json"
with open(result_path , "w") as f:
    import json
    json.dump(result, f, indent=4)  
print(f"Test accuracy: {score}")

Test accuracy: 0.8461538461538461
